In [40]:
import pandas as pd
from scipy import stats
import numpy as np


df = pd.read_csv("cookie_cats.csv")
df

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True
...,...,...,...,...,...
90184,9999441,gate_40,97,True,False
90185,9999479,gate_40,30,False,False
90186,9999710,gate_30,28,True,False
90187,9999768,gate_40,51,True,False


In [41]:
# Data pre processing

# The diagnostic checklist -
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nMissing %:\n", (df.isnull().sum() / len(df) * 100).round(1))
print("\nDuplicate rows:", df.duplicated().sum())

# Check every object column for inconsistency
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col} — unique values sample:")
    print(df[col].unique()[:15])



Shape: (90189, 5)

Dtypes:
 userid             int64
version           object
sum_gamerounds     int64
retention_1         bool
retention_7         bool
dtype: object

Missing values:
 userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

Missing %:
 userid            0.0
version           0.0
sum_gamerounds    0.0
retention_1       0.0
retention_7       0.0
dtype: float64

Duplicate rows: 0

version — unique values sample:
['gate_30' 'gate_40']


In [42]:

print(df['version'].value_counts())
print(df.groupby('version')[['retention_1','retention_7']].mean())

# -------> HYPOTHESIS------------> is there a relation between gate position and retention

# NUll Hypothesis(H0): retention is the same regardless of gate position (no difference)
# Alternate Hypothesis (H1): retention differs between gate_30 and gate_40
# for Day1 and Day7

gate30 = df[df['version'] == 'gate_30']
gate40 = df[df['version'] == 'gate_40']
gate30


version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
         retention_1  retention_7
version                          
gate_30     0.448188     0.190201
gate_40     0.442283     0.182000


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
6,1066,gate_30,0,False,False
11,2101,gate_30,0,False,False
13,2179,gate_30,39,True,False
...,...,...,...,...,...
90179,9998576,gate_30,14,True,False
90180,9998623,gate_30,7,False,False
90182,9999178,gate_30,21,True,False
90183,9999349,gate_30,10,False,False


In [43]:
# Day-1 retention test----> two sample t-test


t_stats1, p_value1 = stats.ttest_ind(gate30['retention_1'], gate40['retention_1'], equal_var=False)
print(f"\nDay-1 retention — p-value: {p_value1:.4f}")


Day-1 retention — p-value: 0.0744


In [44]:
alpha = 0.05
if p_value1 < alpha:
    print("We reject the null hypothesis - there is a  difference")
else:
    print("We accept the null hypothesis - no difference")


We accept the null hypothesis - no difference


In [45]:
# Day-7 retention test----> Two Sample t-test — THIS IS WHERE IT GETS INTERESTING


t_stats7, p_value7 = stats.ttest_ind(gate30['retention_7'], gate40['retention_7'],equal_var=False)
print(f"Day-7 retention — p-value: {p_value7:.4f}")

Day-7 retention — p-value: 0.0016


In [46]:
if p_value7 < alpha:
    print("We reject the null hypothesis - there is a  difference")
else:
    print("We accept the null hypothesis - no difference")

We reject the null hypothesis - there is a  difference


In [49]:
# Confidence interval — how BIG is the difference, not just "is it different"
p1, p2 = gate30['retention_7'].mean(), gate40['retention_7'].mean()
n1, n2 = len(gate30), len(gate40)
diff = p1 - p2
se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)

ci_low, ci_high = diff - 1.96*se, diff + 1.96*se

print(f"Day-7 difference: {diff*100:.2f} percentage points")
print(f"95% CI: [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")



Day-7 difference: 0.82 percentage points
95% CI: [0.31%, 1.33%]


In [50]:
# Outlier check — is one crazy player messing up the results?
print(df['sum_gamerounds'].describe())
print("99th percentile:", df['sum_gamerounds'].quantile(0.99))
print("Max:", df['sum_gamerounds'].max())

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64
99th percentile: 493.0
Max: 49854
